In [1]:
AGENTS = [
    "20250603_Refact_Agent_claude-4-sonnet",
    "20250720_Lingxi-v1.5_claude-4-sonnet-20250514",
    "20250805_openhands-Qwen3-Coder-480B-A35B-Instruct",
    "20250928_trae_doubao_seed_code",
    "20250807_mini-v1.7.0_gpt-5-mini",
]

In [2]:
import json
import os
import pandas as pd
import numpy as np
import pprint as pp
from scipy.stats import wilcoxon
from dataset.extract_ground_truths.effect.process_agent_patch import get_diff_info_per_instance
from execution.util import get_instance_ids

/home/yusuf/explainbench/explainbench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
resolved_dict = {}
base_path = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/efficacy"
for agent in AGENTS:
    path = agent + ".json" if "mini-" not in agent else agent + "_resolved.json"
    path = os.path.join(base_path, path)
    with open(path, "r") as f:
        temp = json.load(f)
    resolved_dict[agent] = temp["resolved"]

# INTENT

In [4]:
def extract_score_local_intent(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx][0])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers

In [5]:
local_intent = "results_intent_local/eval.individual.intent.json"
with open(local_intent, "r") as f:
    local_intent = json.load(f)

In [6]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_intent, agent, "local_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_intent = pd.DataFrame(df_dict)

In [7]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.intent.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)

answers = []
for idx, row in local_intent.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_intent["truth"] = answers

In [8]:
ee_intent = "results_intent_ee/final_results_intent_pbtassertionmcq.json"
with open(ee_intent, "r") as f:
    ee_intent = json.load(f)

In [9]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx]["selection"])
            gts.append(result_dict["answer_gt"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts 

In [10]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts = extract_score_local_ee(ee_intent, agent, "ee_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved,
    "truth": gts
}        

ee_intent = pd.DataFrame(df_dict)

In [11]:
ee_intent

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,ee_intent,1,astropy__astropy-13977,A,False,False,B
1,20250603_Refact_Agent_claude-4-sonnet,ee_intent,2,astropy__astropy-13977,A,False,False,B
2,20250603_Refact_Agent_claude-4-sonnet,ee_intent,3,astropy__astropy-13977,A,False,False,B
3,20250603_Refact_Agent_claude-4-sonnet,ee_intent,4,astropy__astropy-13977,A,False,False,B
4,20250603_Refact_Agent_claude-4-sonnet,ee_intent,5,astropy__astropy-13977,A,False,False,B
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,1,pydata__xarray-6938,A,False,False,D
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,2,pydata__xarray-6938,A,False,False,D
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,3,pydata__xarray-6938,A,False,False,D
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,4,pydata__xarray-6938,B,False,False,D


In [12]:
local_intent["answer"] =local_intent.answer.str.upper()
local_intent["truth"] =local_intent.truth.str.upper()

In [13]:
local_intent.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7425 entries, 0 to 7424
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   agent_name  7425 non-null   object 
 1   q_type      7425 non-null   object 
 2   trial       7425 non-null   int64  
 3   id_         7425 non-null   object 
 4   answer      7425 non-null   object 
 5   score       7425 non-null   float64
 6   resolved    7425 non-null   bool   
 7   truth       7425 non-null   object 
dtypes: bool(1), float64(1), int64(1), object(5)
memory usage: 413.4+ KB


In [14]:
intent = pd.concat([local_intent, ee_intent])
intent

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,local_intent,1,django__django-11179,D,1.0,True,D
1,20250603_Refact_Agent_claude-4-sonnet,local_intent,2,django__django-11179,D,1.0,True,D
2,20250603_Refact_Agent_claude-4-sonnet,local_intent,3,django__django-11179,D,1.0,True,D
3,20250603_Refact_Agent_claude-4-sonnet,local_intent,4,django__django-11179,D,1.0,True,D
4,20250603_Refact_Agent_claude-4-sonnet,local_intent,5,django__django-11179,D,1.0,True,D
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,1,pydata__xarray-6938,A,0.0,False,D
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,2,pydata__xarray-6938,A,0.0,False,D
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,3,pydata__xarray-6938,A,0.0,False,D
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,4,pydata__xarray-6938,B,0.0,False,D


In [15]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    intent.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score_intent = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "intent_score"})
)

agent_score_intent

,agent_name,intent_score
0,20250603_Refact_Agent_claude-4-sonnet,0.538721
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.536364
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.537710
3,20250807_mini-v1.7.0_gpt-5-mini,0.384848
4,20250928_trae_doubao_seed_code,0.484175


# EFFECT

In [16]:
local_effect = "results_effect_local/eval.individual.effect.json"

In [17]:
with open(local_effect, "r") as f:
    local_effect = json.load(f)

In [18]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_effect, agent, "local_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_effect = pd.DataFrame(df_dict)

In [19]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)


answers = []
for idx, row in local_effect.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_effect["truth"] = answers

In [20]:
local_effect["answer"] = local_effect.answer.str.upper()
local_effect["truth"] = local_effect.truth.str.upper()

In [21]:
ee_effect = "results_effect_ee/final_results_effect_pbtresultmcq.json"
with open(ee_effect, "r") as f:
    ee_effect = json.load(f)

In [22]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append([result_dict["all_pred"][idx]["before_selection"], result_dict["all_pred"][idx]["after_selection"]])
            gts.append(result_dict["answer_gt"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts 

In [23]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts = extract_score_local_ee(ee_effect, agent, "ee_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved,
    "truth": gts
}        

ee_effect = pd.DataFrame(df_dict)

In [24]:
ee_effect

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,ee_effect,1,django__django-12304,"[A, A]",False,True,"[D, E]"
1,20250603_Refact_Agent_claude-4-sonnet,ee_effect,2,django__django-12304,"[A, A]",False,True,"[D, E]"
2,20250603_Refact_Agent_claude-4-sonnet,ee_effect,3,django__django-12304,"[A, A]",False,True,"[D, E]"
3,20250603_Refact_Agent_claude-4-sonnet,ee_effect,4,django__django-12304,"[A, A]",False,True,"[D, E]"
4,20250603_Refact_Agent_claude-4-sonnet,ee_effect,5,django__django-12304,"[A, A]",False,True,"[D, E]"
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,1,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,2,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,3,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,4,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"


In [25]:
effect = pd.concat([local_effect, ee_effect])

In [26]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    effect.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score_effect = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "effect_score"})
)

agent_score_effect.round(3)

,agent_name,effect_score
0,20250603_Refact_Agent_claude-4-sonnet,0.602
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.603
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.610
3,20250807_mini-v1.7.0_gpt-5-mini,0.439
4,20250928_trae_doubao_seed_code,0.587


In [27]:
df = pd.concat([effect, intent])

In [28]:
df

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,local_effect,1,django__django-13449,E,0.0,True,D
1,20250603_Refact_Agent_claude-4-sonnet,local_effect,2,django__django-13449,E,0.0,True,D
2,20250603_Refact_Agent_claude-4-sonnet,local_effect,3,django__django-13449,E,0.0,True,D
3,20250603_Refact_Agent_claude-4-sonnet,local_effect,4,django__django-13449,E,0.0,True,D
4,20250603_Refact_Agent_claude-4-sonnet,local_effect,5,django__django-13449,E,0.0,True,D
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,1,pydata__xarray-6938,A,0.0,False,D
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,2,pydata__xarray-6938,A,0.0,False,D
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,3,pydata__xarray-6938,A,0.0,False,D
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,4,pydata__xarray-6938,B,0.0,False,D


In [29]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    df.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "agent_score"})
)

agent_score.round(3)

,agent_name,agent_score
0,20250603_Refact_Agent_claude-4-sonnet,0.570
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.570
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.574
3,20250807_mini-v1.7.0_gpt-5-mini,0.412
4,20250928_trae_doubao_seed_code,0.536


In [30]:
round((agent_score_effect["effect_score"] + agent_score_intent["intent_score"])/2, 3)

0    0.570
1    0.570
2    0.574
3    0.412
4    0.536
dtype: float64

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29700 entries, 0 to 7424
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   agent_name  29700 non-null  object 
 1   q_type      29700 non-null  object 
 2   trial       29700 non-null  int64  
 3   id_         29700 non-null  object 
 4   answer      29700 non-null  object 
 5   score       29700 non-null  float64
 6   resolved    29700 non-null  bool   
 7   truth       29700 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(5)
memory usage: 1.8+ MB


In [32]:
# Basic checks
assert df.shape[1] == 8
assert set(["agent_name","q_type","trial","id_","answer","score","resolved","truth"]).issubset(df.columns)

# Cardinalities
n_agents = df["agent_name"].nunique()
n_ids = df["id_"].nunique()
n_q = df["q_type"].nunique()
n_trials = df["trial"].nunique()

print("agents:", n_agents, "ids:", n_ids, "q_types:", n_q, "trials:", n_trials)

# Expected total rows if complete grid:
expected = n_agents * n_ids * n_q * n_trials
print("expected rows:", expected, "actual rows:", len(df))

# Identify missing combinations (optional but recommended)
grid = (
    df[["agent_name","id_","q_type","trial"]]
    .drop_duplicates()
    .assign(present=True)
)
# If you want to explicitly find missing combos:
full = (
    pd.MultiIndex.from_product(
        [
            df["agent_name"].unique(),
            df["id_"].unique(),
            df["q_type"].unique(),
            df["trial"].unique(),
        ],
        names=["agent_name","id_","q_type","trial"]
    )
    .to_frame(index=False)
)
missing = full.merge(grid, how="left", on=["agent_name","id_","q_type","trial"])
missing = missing[missing["present"].isna()].drop(columns="present")
print("missing rows:", len(missing))


agents: 5 ids: 297 q_types: 4 trials: 5
expected rows: 29700 actual rows: 29700
missing rows: 0


In [33]:
per_item = (
    df.groupby(["agent_name", "id_", "q_type"], as_index=False)
      .agg(
          score_mean=("score", "mean"),
          resolved_any=("resolved", "max"),   # OR use mean if "resolved" varies by trial
          resolved_rate=("resolved", "mean"), # resolution frequency across trials
          n_trials=("trial", "nunique"),
      )
)

In [34]:
per_item

,agent_name,id_,q_type,score_mean,resolved_any,resolved_rate,n_trials
0,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,ee_effect,1.0,True,1.0,5
1,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,ee_intent,1.0,True,1.0,5
2,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,local_effect,1.0,True,1.0,5
3,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,local_intent,0.8,True,1.0,5
4,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13033,ee_effect,0.4,False,0.0,5
...,...,...,...,...,...,...,...
5935,20250928_trae_doubao_seed_code,sympy__sympy-24539,local_intent,0.0,True,1.0,5
5936,20250928_trae_doubao_seed_code,sympy__sympy-24562,ee_effect,1.0,True,1.0,5
5937,20250928_trae_doubao_seed_code,sympy__sympy-24562,ee_intent,1.0,True,1.0,5
5938,20250928_trae_doubao_seed_code,sympy__sympy-24562,local_effect,1.0,True,1.0,5


In [35]:
print(sorted(df["q_type"].unique()))

# Fill this mapping with your exact labels
qtype_to_col = {
    "ee_intent": ("End-to-End", "Intent"),
    "ee_effect": ("End-to-End", "Effect"),
    "local_intent":      ("Local",      "Intent"),
    "local_effect":      ("Local",      "Effect"),
}


['ee_effect', 'ee_intent', 'local_effect', 'local_intent']


In [36]:
# Keep only q_types that are in the mapping (optional but safer)
mapped = per_item[per_item["q_type"].isin(qtype_to_col.keys())].copy()

# A simple agent x q_type mean table
agent_q = (
    mapped.groupby(["agent_name", "q_type"], as_index=False)
          .agg(score=("score_mean", "mean"))
)

# Pivot to columns
agent_pivot = agent_q.pivot(index="agent_name", columns="q_type", values="score")
agent_pivot = agent_pivot.rename(columns={k: f"{v[0]}|{v[1]}" for k, v in qtype_to_col.items()})
agent_pivot = agent_pivot.reindex(columns=[
    "End-to-End|Intent",
    "End-to-End|Effect",
    "Local|Intent",
    "Local|Effect",
])

In [37]:
agent_pivot

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect
agent_name,,,,
20250603_Refact_Agent_claude-4-sonnet,0.722559,0.713805,0.354882,0.489562
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.704377,0.715152,0.368350,0.491582
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.722559,0.697643,0.352862,0.521886
20250807_mini-v1.7.0_gpt-5-mini,0.467340,0.507744,0.302357,0.370370
20250928_trae_doubao_seed_code,0.636364,0.716498,0.331987,0.457239


In [38]:
agent_pivot["Expl. Score"] = agent_pivot.mean(axis=1, skipna=True)

In [39]:
agent_pivot.round(3)

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect,Expl. Score
agent_name,,,,,
20250603_Refact_Agent_claude-4-sonnet,0.723,0.714,0.355,0.490,0.570
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.704,0.715,0.368,0.492,0.570
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.723,0.698,0.353,0.522,0.574
20250807_mini-v1.7.0_gpt-5-mini,0.467,0.508,0.302,0.370,0.412
20250928_trae_doubao_seed_code,0.636,0.716,0.332,0.457,0.536


In [40]:
per_instance = (
    df.groupby(
        ["agent_name", "id_"],
        as_index=False
    )["score"]
    .mean()
    .rename(columns={"score": "explanation_score"})
)

In [41]:
per_instance

,agent_name,id_,explanation_score
0,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,0.95
1,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13033,0.35
2,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13236,0.10
3,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13453,0.45
4,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13579,0.30
...,...,...,...
1480,20250928_trae_doubao_seed_code,sympy__sympy-24066,0.70
1481,20250928_trae_doubao_seed_code,sympy__sympy-24213,0.05
1482,20250928_trae_doubao_seed_code,sympy__sympy-24443,0.50
1483,20250928_trae_doubao_seed_code,sympy__sympy-24539,0.50


In [42]:
subset = per_instance[
    (per_instance["agent_name"].isin(["20250928_trae_doubao_seed_code", "20250805_openhands-Qwen3-Coder-480B-A35B-Instruct"]))
]

paired = (
    subset.pivot(
        index="id_",
        columns="agent_name",
        values="explanation_score"
    )
    .dropna()
)

In [43]:
paired

agent_name,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,20250928_trae_doubao_seed_code
id_,,
astropy__astropy-12907,0.50,0.40
astropy__astropy-13033,0.55,1.00
astropy__astropy-13236,0.25,0.25
astropy__astropy-13453,0.50,0.60
astropy__astropy-13579,0.25,0.25
...,...,...
sympy__sympy-24066,0.55,0.70
sympy__sympy-24213,1.00,0.05
sympy__sympy-24443,0.35,0.50


In [44]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(
    paired["20250928_trae_doubao_seed_code"],
    paired["20250805_openhands-Qwen3-Coder-480B-A35B-Instruct"],
    alternative="two-sided"
)

print(f"W={stat:.3f}, p={p_value:.4g}, N={len(paired)}")

W=10356.500, p=0.001464, N=297


In [45]:
per_instance = (
    df.groupby(
        ["agent_name", "trial"],
        as_index=False
    )["score"]
    .mean()
    .rename(columns={"score": "explanation_score"})
)

In [46]:
per_instance

,agent_name,trial,explanation_score
0,20250603_Refact_Agent_claude-4-sonnet,1,0.553872
1,20250603_Refact_Agent_claude-4-sonnet,2,0.572391
2,20250603_Refact_Agent_claude-4-sonnet,3,0.568182
3,20250603_Refact_Agent_claude-4-sonnet,4,0.568182
4,20250603_Refact_Agent_claude-4-sonnet,5,0.588384
5,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,1,0.556397
6,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,2,0.565657
7,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,3,0.564815
8,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,4,0.579966
9,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,5,0.582492


In [47]:
per_instance.groupby("agent_name")["explanation_score"].std()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.012351
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.011023
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.007401
20250807_mini-v1.7.0_gpt-5-mini                      0.012663
20250928_trae_doubao_seed_code                       0.011378
Name: explanation_score, dtype: float64

In [48]:
per_instance.groupby("agent_name")["explanation_score"].mean()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.570202
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.569865
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.573737
20250807_mini-v1.7.0_gpt-5-mini                      0.411953
20250928_trae_doubao_seed_code                       0.535522
Name: explanation_score, dtype: float64

In [49]:
per_instance.groupby("agent_name")["explanation_score"].sem()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.005524
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.004930
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.003310
20250807_mini-v1.7.0_gpt-5-mini                      0.005663
20250928_trae_doubao_seed_code                       0.005088
Name: explanation_score, dtype: float64

In [50]:
agent_pivot["sem"] = per_instance.groupby("agent_name")["explanation_score"].sem()

In [51]:
agent_pivot.round(4)

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect,Expl. Score,sem
agent_name,,,,,,
20250603_Refact_Agent_claude-4-sonnet,0.7226,0.7138,0.3549,0.4896,0.5702,0.0055
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.7044,0.7152,0.3684,0.4916,0.5699,0.0049
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.7226,0.6976,0.3529,0.5219,0.5737,0.0033
20250807_mini-v1.7.0_gpt-5-mini,0.4673,0.5077,0.3024,0.3704,0.4120,0.0057
20250928_trae_doubao_seed_code,0.6364,0.7165,0.3320,0.4572,0.5355,0.0051
